# Эксперимент B: shared encoder + embeddings расстояний

Перед запуском прикрепите Kaggle Dataset, созданный `build_kaggle_dataset.ipynb`. Все четыре изображения обрабатываются одним shared encoder, признаки получают embeddings физических расстояний и объединяются `1×1`-свёртками на каждом масштабе. Transformer, Fourier-ветвь и unit-complex head намеренно не используются, чтобы изолировать гипотезу B.

In [ ]:
import csv
import json
import math
import os
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    from pytorch_msssim import ssim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])
    from pytorch_msssim import ssim

# Change this only when your Kaggle Dataset has a different slug.
KAGGLE_DATASET_DIR = Path('/kaggle/input/ai-phonetics-data-v1')
WORK_DIR = Path('/kaggle/working/experiment_b_run')
IMAGE_SIZE = 128
WAVELENGTH = 532e-9
PIXEL_PITCH = 10e-6
ZS = (24.1e-3, 25.3e-3, 26.5e-3, 27.7e-3)
SEED = 42
BATCH_SIZE = 16
EPOCHS = 40
WARMUP_EPOCHS = 2
PATIENCE = 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = min(4, os.cpu_count() or 1)

def find_dataset_root(path: Path) -> Path:
    if (path / 'dataset_metadata.json').is_file():
        return path
    matches = list(path.rglob('dataset_metadata.json'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Cannot find exactly one dataset_metadata.json below {path}')
    return matches[0].parent

DATA_ROOT = find_dataset_root(KAGGLE_DATASET_DIR)
BSDS_ROOT = DATA_ROOT / 'BSDS500' / 'data'
WORK_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)
print('device =', 'cuda' if torch.cuda.is_available() else 'cpu')


## Загрузчик манифестов

Синтетические каналы восстанавливаются из seed, а натуральные — из исходного JPEG и координат кропа. Каждый канал нормируется на собственный максимум, как требует уточнение организаторов.

In [ ]:
def normalise(image, eps=1e-8):
    maximum = image.amax()
    if maximum <= eps:
        raise ValueError('All-black image cannot be normalized')
    return image / maximum

def to_tensor(image):
    return normalise(torch.from_numpy(np.asarray(image.convert('L'), dtype=np.float32) / 255.0))

def draw_glyph(canvas, rng, value):
    char = '0123456789ABCDEFGHJKLMNPQRSTUVWXYZ'[int(rng.integers(0, 34))]
    font_size = int(rng.integers(22, 66))
    try:
        font = ImageFont.truetype('DejaVuSans.ttf', size=font_size)
    except OSError:
        font = ImageFont.load_default()
    glyph = Image.new('L', (font_size * 2, font_size * 2), 0)
    glyph_draw = ImageDraw.Draw(glyph)
    box = glyph_draw.textbbox((0, 0), char, font=font)
    glyph_draw.text((-box[0], -box[1]), char, fill=value, font=font)
    box = glyph.getbbox()
    if box is not None:
        glyph = glyph.crop(box).rotate(float(rng.uniform(-25, 25)), resample=Image.Resampling.BILINEAR, expand=True)
        canvas.paste(glyph, (int(rng.integers(0, max(1, 129 - glyph.width))), int(rng.integers(0, max(1, 129 - glyph.height)))), glyph)

def synthetic(seed):
    rng = np.random.default_rng(int(seed))
    canvas = Image.new('L', (128, 128), 0)
    draw = ImageDraw.Draw(canvas)
    for _ in range(int(rng.integers(1, 5))):
        kind = str(rng.choice(['ellipse', 'rectangle', 'line', 'ring', 'point', 'glyph']))
        value, width = int(rng.integers(80, 256)), int(rng.integers(1, 7))
        x0, x1 = sorted(rng.integers(4, 124, size=2).tolist()); y0, y1 = sorted(rng.integers(4, 124, size=2).tolist())
        x1, y1 = min(127, max(x1, x0 + 8)), min(127, max(y1, y0 + 8))
        if kind == 'ellipse': draw.ellipse((x0, y0, x1, y1), fill=value if rng.random() < .5 else None, outline=value, width=width)
        elif kind == 'rectangle': draw.rectangle((x0, y0, x1, y1), fill=value if rng.random() < .5 else None, outline=value, width=width)
        elif kind == 'line': draw.line((x0, y0, x1, y1), fill=value, width=width)
        elif kind == 'ring': draw.ellipse((x0, y0, x1, y1), outline=value, width=width)
        elif kind == 'point':
            radius = int(rng.integers(2, 10)); cx, cy = int(rng.integers(radius, 128 - radius)), int(rng.integers(radius, 128 - radius))
            draw.ellipse((cx - radius, cy - radius, cx + radius, cy + radius), fill=value)
        else: draw_glyph(canvas, rng, value)
    return to_tensor(canvas)

class QuadDataset(Dataset):
    def __init__(self, manifest):
        self.records = [json.loads(line) for line in Path(manifest).read_text(encoding='utf-8').splitlines() if line]
    def __len__(self): return len(self.records)
    def __getitem__(self, index):
        images = []
        for item in self.records[index]['items']:
            if item['source'] == 'synthetic':
                images.append(synthetic(item['seed']))
            else:
                path = BSDS_ROOT / item['path']
                with Image.open(path) as source:
                    x, y = item['crop']['x'], item['crop']['y']
                    images.append(to_tensor(source.crop((x, y, x + 128, y + 128))))
        quad = torch.stack(images)
        assert quad.shape == (4, 128, 128)
        return quad

train_set = QuadDataset(DATA_ROOT / 'manifests' / 'train.jsonl')
val_set = QuadDataset(DATA_ROOT / 'manifests' / 'val.jsonl')
print(f'train={len(train_set)}, val={len(val_set)}, example={tuple(train_set[0].shape)}')


## Shared-depth U-Net, слой Френеля и loss

Encoder имеет общие веса для всех плоскостей. Нормированные физические расстояния кодируются отдельным MLP на каждом масштабе. После этого четыре набора признаков объединяются обычной `1×1`-свёрткой; attention между плоскостями в этом эксперименте отсутствует.

In [ ]:
class DoubleConv(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU(), nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.GroupNorm(8, cout), nn.SiLU())

class UpBlock(nn.Module):
    def __init__(self, cin, skip, cout):
        super().__init__(); self.reduce = nn.Conv2d(cin, cout, 1); self.block = DoubleConv(cout + skip, cout)
    def forward(self, x, skip):
        x = nn.functional.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        return self.block(torch.cat((self.reduce(x), skip), dim=1))

class DepthEmbedding(nn.Module):
    def __init__(self, channels):
        super().__init__()
        hidden = max(16, channels // 2)
        self.net = nn.Sequential(nn.Linear(1, hidden), nn.SiLU(), nn.Linear(hidden, channels))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
    def forward(self, depths):
        return self.net(depths[:, None])[:, :, None, None]

class PlaneFusion(nn.Sequential):
    def __init__(self, channels, planes=4):
        super().__init__(nn.Conv2d(planes * channels, channels, 1, bias=False), nn.GroupNorm(8, channels), nn.SiLU())

class SharedDepthUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.planes = 4
        channels = (32, 64, 128, 256, 512)
        self.encoders = nn.ModuleList([DoubleConv(1, 32), DoubleConv(32, 64), DoubleConv(64, 128), DoubleConv(128, 256), DoubleConv(256, 512)])
        self.depth_embeddings = nn.ModuleList([DepthEmbedding(c) for c in channels])
        self.fusions = nn.ModuleList([PlaneFusion(c, self.planes) for c in channels])
        self.pool = nn.MaxPool2d(2)
        self.d3, self.d2, self.d1, self.d0 = UpBlock(512, 256, 256), UpBlock(256, 128, 128), UpBlock(128, 64, 64), UpBlock(64, 32, 32)
        self.head = nn.Conv2d(32, 1, 1)
        z_mm = torch.tensor(ZS, dtype=torch.float32) * 1e3
        self.register_buffer('depths', (z_mm - z_mm.mean()) / (z_mm.max() - z_mm.min()))
    def _fuse(self, features, fusion, batch_size):
        _, channels, height, width = features.shape
        features = features.reshape(batch_size, self.planes, channels, height, width)
        return fusion(features.reshape(batch_size, self.planes * channels, height, width))
    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != self.planes:
            raise ValueError(f'Expected (B, 4, H, W), got {tuple(x.shape)}')
        batch_size, _, height, width = x.shape
        features = x.reshape(batch_size * self.planes, 1, height, width)
        depths = self.depths.repeat(batch_size)
        fused = []
        for level, (encoder, embedding, fusion) in enumerate(zip(self.encoders, self.depth_embeddings, self.fusions)):
            features = encoder(features)
            features = features + embedding(depths).to(features.dtype)
            fused.append(self._fuse(features, fusion, batch_size))
            if level + 1 < len(self.encoders):
                features = self.pool(features)
        e0, e1, e2, e3, bottleneck = fused
        x = self.d3(bottleneck, e3); x = self.d2(x, e2); x = self.d1(x, e1); x = self.d0(x, e0)
        return math.pi * torch.tanh(self.head(x))

class FresnelLayer(nn.Module):
    def __init__(self, device):
        super().__init__(); c = (torch.arange(128, dtype=torch.float64) - 64) * PIXEL_PITCH; u, v = torch.meshgrid(c, c, indexing='ij')
        kernels = [torch.exp(1j * math.pi * (u.square() + v.square()) / (WAVELENGTH * z)).to(torch.complex64) for z in ZS]
        self.register_buffer('kernels', torch.stack(kernels).to(device), persistent=False)
    def forward(self, phase):
        field = torch.polar(torch.ones_like(phase, dtype=torch.float32), phase.float())
        outputs = []
        for kernel in self.kernels:
            propagated = torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(field * kernel, dim=(-2, -1))), dim=(-2, -1)).abs()
            outputs.append(propagated)
        amplitude = torch.cat(outputs, dim=1)
        return amplitude / (amplitude.amax(dim=(-2, -1), keepdim=True) + 1e-8)

def pearson_cc(a, b, eps=1e-8):
    a = a - a.mean(dim=(-2, -1), keepdim=True); b = b - b.mean(dim=(-2, -1), keepdim=True)
    return (a * b).sum(dim=(-2, -1)) / (torch.sqrt(a.square().sum(dim=(-2, -1)) * b.square().sum(dim=(-2, -1))) + eps)

def doe_loss(target, reconstruction):
    cc = pearson_cc(target, reconstruction)
    ssim_values = torch.stack([ssim(target[:, m:m+1], reconstruction[:, m:m+1], data_range=1.0, size_average=False) for m in range(4)], dim=1)
    return (1 - cc).mean() + 1.1 * (1 - ssim_values).mean(), cc, ssim_values


## Обучение

Физический слой и loss всегда выполняются в `float32`; AMP применяется только к сети. Сохраняются checkpoints с лучшим `val_loss`, средним КК и средним ИСС, а validation-метрики логируются отдельно по каждой плоскости.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pin_memory = device.type == 'cuda'
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0)
model, physics = SharedDepthUNet().to(device), FresnelLayer(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f'model parameters: {parameter_count:,}')
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
def multiplier(epoch):
    if epoch < WARMUP_EPOCHS: return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return (1e-5 / LR) + (1 - 1e-5 / LR) * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

@torch.no_grad()
def validate():
    model.eval(); loss_total = 0.0; cc_total = torch.zeros(4, device=device); ssim_total = torch.zeros(4, device=device); count = 0
    for target in val_loader:
        target = target.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            phase = model(2 * target - 1)
        reconstruction = physics(phase.float())
        loss, cc, ssim_values = doe_loss(target, reconstruction)
        loss_total += loss.item() * target.shape[0]
        cc_total += cc.sum(dim=0); ssim_total += ssim_values.sum(dim=0); count += target.shape[0]
    return loss_total / count, (cc_total / count).cpu().tolist(), (ssim_total / count).cpu().tolist()

history, best_loss, best_cc, best_ssim, stale = [], float('inf'), -float('inf'), -float('inf'), 0
for epoch in range(EPOCHS):
    model.train(); running, seen = 0.0, 0
    for target in train_loader:
        target = target.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            phase = model(2 * target - 1)
        with torch.autocast(device_type=device.type, enabled=False):
            loss, _, _ = doe_loss(target, physics(phase.float()))
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        running += loss.detach().item() * target.shape[0]; seen += target.shape[0]
    val_loss, val_cc_planes, val_ssim_planes = validate(); scheduler.step()
    val_cc, val_ssim = float(np.mean(val_cc_planes)), float(np.mean(val_ssim_planes))
    row = {'epoch': epoch + 1, 'train_loss': running / seen, 'val_loss': val_loss, 'val_cc': val_cc, 'val_ssim': val_ssim, 'lr': optimizer.param_groups[0]['lr']}
    row.update({f'val_cc_z{m+1}': value for m, value in enumerate(val_cc_planes)})
    row.update({f'val_ssim_z{m+1}': value for m, value in enumerate(val_ssim_planes)})
    history.append(row); print(row)
    torch.save({'epoch': epoch + 1, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'metrics': row}, WORK_DIR / 'last.pt')
    if val_loss < best_loss:
        best_loss, stale = val_loss, 0
        torch.save({'epoch': epoch + 1, 'model': model.state_dict(), 'metrics': row}, WORK_DIR / 'best.pt')
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f'Early stopping after {epoch + 1} epochs'); break
    if val_cc > best_cc:
        best_cc = val_cc
        torch.save({'epoch': epoch + 1, 'model': model.state_dict(), 'metrics': row}, WORK_DIR / 'best_cc.pt')
    if val_ssim > best_ssim:
        best_ssim = val_ssim
        torch.save({'epoch': epoch + 1, 'model': model.state_dict(), 'metrics': row}, WORK_DIR / 'best_ssim.pt')

with (WORK_DIR / 'metrics.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=history[0].keys()); writer.writeheader(); writer.writerows(history)
(WORK_DIR / 'config.json').write_text(json.dumps({'experiment': 'B_shared_encoder_depth_embeddings', 'image_size': IMAGE_SIZE, 'wavelength': WAVELENGTH, 'pixel_pitch': PIXEL_PITCH, 'zs': ZS, 'batch_size': BATCH_SIZE, 'epochs_requested': EPOCHS, 'seed': SEED, 'parameter_count': parameter_count}, indent=2))
print('Saved:', WORK_DIR / 'best.pt', WORK_DIR / 'best_cc.pt', WORK_DIR / 'best_ssim.pt', WORK_DIR / 'metrics.csv')
